# 🧠 DSA Lab 14: Binary Search Trees: Search, Insert, Delete and Balance
### Student Notebook

**Data Structures & Algorithms Lab Manual | University of Okara | Fall 2026**

Prepared by **Muhammad Ramzan**<br>
MPhil Scholar (Artificial Intelligence), PUCIT, University of the Punjab, Lahore<br>
Lab Engineer, Department of Computer Science, University of Okara

🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)

---
**How to use:** run each code cell with **Shift + Enter**, from top to bottom. Read the matching section in the Lab 14 manual, change values and re-run to see what happens.

> 📝 **Your job:** run the examples, then complete every `# TODO` in the Lab Tasks part. Each task has a **test cell**: when it prints ✅ your solution is correct. Write pseudocode as comments before coding!

## 📖 Part A: The BST Property

A **Binary Search Tree** is a binary tree where, for **every** node:

- All keys in the **left subtree** are **smaller** than the node’s key.
- All keys in the **right subtree** are **greater** than the node’s key.
- Both subtrees are also BSTs. (In this lab, **no duplicate keys**.)

**A valid BST**

```text
            50
          /    \
        30      70
       /  \    /  \
     20   40  60   80
         /
       35
Inorder: 20 30 35 40 50 60 70 80   (sorted!)
```

> **⚠️ A common trap**
>
> It is **not enough** that each node is bigger than its left child and smaller than its right child. Suppose 55 is placed as the right child of 40 in the tree above. Locally 40 < 55 looks fine, but 55 is in the **left subtree of 50** while being greater than 50, so the tree is **not** a BST. Every key must respect **all** of its ancestors.

## 🔎 Part B: Search, Insert, Min and Max

### Search

Compare the key with the current node: equal → found; smaller → go left; greater → go right; reach `None` → not present. At each step we discard one subtree, like binary search.

In [ ]:
class Node:
    def __init__(self, key, left=None, right=None):
        self.key = key
        self.left = left
        self.right = right

def insert(root, key):
    """Insert key into the BST and return the (possibly new) root."""
    if root is None:
        return Node(key)
    if key < root.key:
        root.left = insert(root.left, key)
    elif key > root.key:
        root.right = insert(root.right, key)
    return root                         # duplicates are ignored

def build_bst(keys):
    root = None
    for k in keys:
        root = insert(root, k)
    return root

def inorder(root):
    return inorder(root.left) + [root.key] + inorder(root.right) if root else []

def search(root, key):                  # recursive
    if root is None or root.key == key:
        return root
    return search(root.left, key) if key < root.key else search(root.right, key)

def search_iter(root, key):             # iterative: O(1) extra space
    while root and root.key != key:
        root = root.left if key < root.key else root.right
    return root

root = build_bst([50, 30, 70, 20, 40, 60, 80, 35])
print("inorder:", inorder(root))
print("search 35:", search(root, 35) is not None, "| search 65:", search_iter(root, 65) is not None)

### Insert

Search for the key; the place where the search falls off the tree (`None`) is exactly where the new node belongs. New keys are always inserted as **leaves**.

| Insert | Path | Placed as |
|---|---|---|
| 50 | - | root |
| 30 | 30 < 50 → left | left of 50 |
| 70 | 70 > 50 → right | right of 50 |
| 40 | 40 < 50 → left, 40 > 30 → right | right of 30 |
| 35 | 35 < 50, > 30, < 40 | left of 40 |

### Minimum and Maximum

In [ ]:
def find_min(root):                     # keep going left
    while root.left:
        root = root.left
    return root.key

def find_max(root):                     # keep going right
    while root.right:
        root = root.right
    return root.key

print("min:", find_min(root), "| max:", find_max(root))

## ✂️ Part C: Deletion (Three Cases)

| Case | What to do | Example (tree above) |
|---|---|---|
| 1. Leaf node | Simply remove it | Delete 20 |
| 2. One child | Replace the node with its only child | Delete 40 → 35 takes its place |
| 3. Two children | Replace the key with its **inorder successor** (smallest key in the right subtree), then delete the successor from the right subtree | Delete 50 → 60 becomes the root |

**Case 3 step by step**

```text
Delete 50 (two children):
            50                          60
          /    \                      /    \
        30      70        ->         30      70
       /  \    /  \                 /  \       \
     20   40  60   80              20   40      80
         /                             /
       35                            35
successor = smallest in right subtree = 60
```

In [ ]:
def delete(root, key):
    if root is None:
        return None
    if key < root.key:
        root.left = delete(root.left, key)
    elif key > root.key:
        root.right = delete(root.right, key)
    else:                                   # found the node to delete
        if root.left is None:               # case 1 or 2
            return root.right
        if root.right is None:              # case 2
            return root.left
        succ = root.right                   # case 3: inorder successor
        while succ.left:
            succ = succ.left
        root.key = succ.key
        root.right = delete(root.right, succ.key)
    return root

r = build_bst([50, 30, 70, 20, 40, 60, 80, 35])
for k in [20, 40, 50]:
    r = delete(r, k)
    print(f"after deleting {k}: inorder = {inorder(r)}, root = {r.key}")

## ⚖️ Part D: Balanced vs Skewed Trees

All BST operations walk one path from the root, so they cost **O(h)** where h is the height. The height depends on the **order of insertion**:

```text
Insert 4, 2, 6, 1, 3, 5, 7          Insert 1, 2, 3, 4, 5, 6, 7 (sorted!)
          4                          1
        /   \                         \
       2     6                         2
      / \   / \                         \
     1   3 5   7                         3 ... 7
height 2 = O(log n)                 height 6 = O(n): a linked list!
```

In [ ]:
import random, sys
sys.setrecursionlimit(10_000)

def height(node):
    return -1 if node is None else 1 + max(height(node.left), height(node.right))

n = 1000
keys = list(range(n))
print("sorted insert height  :", height(build_bst(keys)))
random.shuffle(keys)
print("random insert height  :", height(build_bst(keys)))
import math
print("best possible (log2 n):", math.floor(math.log2(n)))

### Self-Balancing Trees: AVL Rotations

An **AVL tree** keeps the **balance factor** = height(left) − height(right) in {−1, 0, +1} at every node. After an insert or delete, if a node becomes unbalanced, a **rotation** fixes it in O(1). This guarantees height O(log n).

**Right rotation**

```text
Right rotation at z (left-left case):

        z                     y
       / \                  /   \
      y   T4     ->         x     z
     / \                  / \   / \
    x   T3               T1 T2 T3  T4
   / \
  T1  T2
```

In [ ]:
def rotate_right(z):
    y = z.left
    z.left = y.right          # T3 moves under z
    y.right = z
    return y                  # y is the new root of this subtree

def rotate_left(z):
    y = z.right
    z.right = y.left
    y.left = z
    return y

skewed = build_bst([30, 20, 10])            # a left-left chain
balanced = rotate_right(skewed)
print("root:", balanced.key, "| left:", balanced.left.key, "| right:", balanced.right.key)

| Imbalance case | Fix |
|---|---|
| Left-Left | Right rotation |
| Right-Right | Left rotation |
| Left-Right | Left rotation on child, then right rotation |
| Right-Left | Right rotation on child, then left rotation |

## ⚙️ C++ Corner

**C++**

```cpp
#include <iostream>
#include <map>
#include <set>
using namespace std;

struct Node { int key; Node *left = nullptr, *right = nullptr; Node(int k) : key(k) {} };

Node* insert(Node* root, int key) {
    if (!root) return new Node(key);
    if (key < root->key) root->left = insert(root->left, key);
    else if (key > root->key) root->right = insert(root->right, key);
    return root;
}

bool search(Node* root, int key) {
    while (root && root->key != key) root = key < root->key ? root->left : root->right;
    return root != nullptr;
}

int main() {
    Node* root = nullptr;
    for (int k : {50, 30, 70, 20, 40, 60, 80}) root = insert(root, k);
    cout << search(root, 60) << search(root, 65) << endl;      // 10

    set<int> s = {50, 30, 70, 20};          // balanced BST (Red-Black tree)
    auto it = s.lower_bound(35);            // smallest key >= 35  -> 50
    cout << *it << endl;
    map<string, double> cgpa = {{"Sara", 3.9}, {"Ali", 3.4}};
    for (auto& [name, g] : cgpa) cout << name << " " << g << endl;   // sorted by name
}
```

## 🧪 Lab Tasks
Difficulty: 🟢 Easy · 🟡 Medium · 🔴 Hard

### Task 1 🟢 Validate a BST
Write `is_bst(root)` that checks the **full** BST property. Pass down the allowed range `(low, high)` for each subtree.

The notebook provides `Node`, `insert`, `build_bst` and `inorder`.

```text
Tree 50 → (30 → right 55), 70  → False  (55 is in the left subtree of 50)
```

In [ ]:
class Node:
    def __init__(self, key, left=None, right=None):
        self.key = key
        self.left = left
        self.right = right

def insert(root, key):
    """Insert key into the BST and return the (possibly new) root."""
    if root is None:
        return Node(key)
    if key < root.key:
        root.left = insert(root.left, key)
    elif key > root.key:
        root.right = insert(root.right, key)
    return root                         # duplicates are ignored

def build_bst(keys):
    root = None
    for k in keys:
        root = insert(root, k)
    return root

def inorder(root):
    return inorder(root.left) + [root.key] + inorder(root.right) if root else []

def is_bst(root, low=float("-inf"), high=float("inf")):
    # TODO: check low < key < high, then recurse with tighter bounds
    pass

In [ ]:
# Test cell: run after your solution
assert is_bst(build_bst([50, 30, 70, 20, 40, 60, 80])) is True
bad = Node(50, Node(30, None, Node(55)), Node(70))
assert is_bst(bad) is False
assert is_bst(None) is True
assert is_bst(Node(5, Node(5))) is False
print("✅ Task 1 passed")

### Task 2 🟢 k-th Smallest Mark
Marks are stored in a BST. Write `kth_smallest(root, k)` using an inorder traversal that **stops early** once the k-th element is reached (do not build the whole list).

```text
BST of [50, 30, 70, 20, 40, 60, 80], k = 3 → 40
```

In [ ]:
def kth_smallest(root, k):
    # TODO: iterative inorder; count visited nodes
    pass

In [ ]:
# Test cell: run after your solution
t = build_bst([50, 30, 70, 20, 40, 60, 80])
assert [kth_smallest(t, k) for k in range(1, 8)] == [20, 30, 40, 50, 60, 70, 80]
assert kth_smallest(t, 8) is None
print("✅ Task 2 passed")

### Task 3 🟡 Range Query: Students Between Two CGPAs
Keys are CGPA × 100 (integers). Write `range_keys(root, low, high)` that returns all keys with `low ≤ key ≤ high` in sorted order. **Prune**: do not visit the left subtree if `root.key < low`, nor the right subtree if `root.key > high`.

```text
keys [310, 250, 365, 290, 330, 390, 280], range [280, 340] → [280, 290, 310, 330]
```

In [ ]:
def range_keys(root, low, high):
    out = []
    # TODO: pruned inorder traversal
    return out

In [ ]:
# Test cell: run after your solution
t = build_bst([310, 250, 365, 290, 330, 390, 280])
assert range_keys(t, 280, 340) == [280, 290, 310, 330]
assert range_keys(t, 400, 500) == [] and range_keys(None, 0, 9) == []
assert range_keys(t, 0, 1000) == inorder(t)
print("✅ Task 3 passed")

### Task 4 🟡 Floor and Ceiling
`floor(root, x)` = largest key ≤ x; `ceiling(root, x)` = smallest key ≥ x (return `None` if none). Solve both in O(h) **without** traversing the whole tree.

Use: a bus timetable stored as minutes after midnight; find the last bus before you arrive and the next bus after you arrive.

```text
times [480, 540, 600, 720, 840], x = 610 → floor 600, ceiling 720
```

In [ ]:
def floor(root, x):
    best = None
    # TODO
    return best

def ceiling(root, x):
    best = None
    # TODO
    return best

In [ ]:
# Test cell: run after your solution
t = build_bst([600, 480, 720, 540, 840])
assert floor(t, 610) == 600 and ceiling(t, 610) == 720
assert floor(t, 400) is None and ceiling(t, 900) is None
assert floor(t, 540) == 540 and ceiling(t, 541) == 600
print("✅ Task 4 passed")

### Task 5 🟡 Lowest Common Ancestor in a BST
The **lowest common ancestor (LCA)** of two keys is the deepest node that has both keys in its subtree (a node can be its own ancestor). In a BST, use the ordering: if both keys are smaller go left, if both are bigger go right, otherwise the current node is the answer. O(h).

```text
BST [6, 2, 8, 0, 4, 7, 9, 3, 5]: lca(2, 8) → 6, lca(2, 4) → 2, lca(3, 5) → 4
```

In [ ]:
def lca(root, a, b):
    # TODO
    pass

In [ ]:
# Test cell: run after your solution
t = build_bst([6, 2, 8, 0, 4, 7, 9, 3, 5])
assert lca(t, 2, 8) == 6 and lca(t, 2, 4) == 2 and lca(t, 3, 5) == 4 and lca(t, 7, 9) == 8
assert lca(None, 1, 2) is None
print("✅ Task 5 passed")

### Task 6 🔴 Sorted Array to Balanced BST
Given sorted roll numbers, build a **height-balanced** BST: make the middle element the root, then recursively build the left half and right half. Return the root.

Verify that the height is ⌊log₂ n⌋ for n = 1,023 and that inorder gives back the array.

```text
[1, 2, 3, 4, 5, 6, 7] → root 4, left subtree root 2, right subtree root 6
```

In [ ]:
def sorted_to_bst(arr, lo=0, hi=None):
    if hi is None:
        hi = len(arr) - 1
    # TODO: middle element is the root; recurse on both halves
    pass

In [ ]:
# Test cell: run after your solution
def h(n): return -1 if n is None else 1 + max(h(n.left), h(n.right))
t = sorted_to_bst([1, 2, 3, 4, 5, 6, 7])
assert t.key == 4 and t.left.key == 2 and t.right.key == 6
big = list(range(1023)); tb = sorted_to_bst(big)
assert inorder(tb) == big and h(tb) == 9 and is_bst(tb)
assert sorted_to_bst([]) is None
print("✅ Task 6 passed")

### Task 7 🔴 A Complete BST Map Class
Build a class `BSTMap` storing **key → value** pairs with methods `put(key, value)` (insert or update), `get(key)` (or `None`), `remove(key)`, `__len__`, `items()` (sorted list of pairs) and `height()`.

Use it as a sorted phone book. Internally, store `key` and `value` in each node and reuse the deletion logic from Part C.

```text
put("Sara", "0300"); put("Ali", "0321"); items() → [("Ali", "0321"), ("Sara", "0300")]
```

In [ ]:
class MapNode:
    def __init__(self, key, value):
        self.key, self.value, self.left, self.right = key, value, None, None

class BSTMap:
    def __init__(self):
        self.root, self.size = None, 0

    def put(self, key, value):
        # TODO: insert or update; increase size only for new keys
        pass

    def get(self, key):
        # TODO
        pass

    def remove(self, key):
        # TODO: return True if removed, False if key not found
        pass

    def __len__(self):
        return self.size

    def items(self):
        # TODO: inorder list of (key, value)
        pass

    def height(self):
        # TODO
        pass

In [ ]:
# Test cell: run after your solution
m = BSTMap()
for k, v in [("Sara", "0300"), ("Ali", "0321"), ("Zoya", "0345"), ("Hamza", "0333")]:
    m.put(k, v)
m.put("Ali", "0322")
assert len(m) == 4 and m.get("Ali") == "0322" and m.get("Nobody") is None
assert m.items() == [("Ali", "0322"), ("Hamza", "0333"), ("Sara", "0300"), ("Zoya", "0345")]
assert m.remove("Sara") is True and m.remove("Sara") is False and len(m) == 3
assert [k for k, _ in m.items()] == ["Ali", "Hamza", "Zoya"] and m.height() >= 1
print("✅ Task 7 passed")

## 🔬 Bonus: Visualise BST Growth (Sorted vs Random Insert)

In [ ]:
# Bonus: run after completing the tasks
import random

def show(node, prefix="", is_left=True, out=None):
    if out is None:
        out = []
    if node:
        show(node.right, prefix + ("│   " if is_left else "    "), False, out)
        out.append(prefix + ("└── " if is_left else "┌── ") + str(node.key))
        show(node.left, prefix + ("    " if is_left else "│   "), True, out)
    return out

print("Random order:")
keys = random.sample(range(1, 16), 15)
print("\n".join(show(build_bst(keys))))
print("\nSorted order (degenerate):")
print("\n".join(show(build_bst(range(1, 8)))))

---
## 🎉 Lab 14 Complete!

Next: **Lab 15: Heaps and Priority Queues**

✅ Upload this notebook to your GitHub repository `DSA-Labs/Lab14/`.

---
*Prepared by **Muhammad Ramzan**, MPhil Scholar (AI), PUCIT, University of the Punjab*<br>
🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)